# Backbone Comparison Experiments

Which backbone architecture provides the best mAP-efficiency tradeoff for jaguar re-identification?

**Tested Backbones:**
- **DINOv3-Large** (vit_large_patch16_dinov3.lvd1689m, 1024-dim, latest self-supervised ViT)
- **DINOv3-Base** (vit_base_patch16_dinov3.lvd1689m, 768-dim, efficient DINOv3 variant)
- **MiewID-MSv2** (conservationxlabs/miewid-msv2, 2152-dim, wildlife re-ID specialist)
- **MiewID-MSv3** (conservationxlabs/miewid-msv3, 2152-dim, wildlife re-ID specialist)
- **DINOv2-Large** (1024-dim, previous-generation self-supervised ViT)
- **DINOv2-Base** (768-dim, balanced)
- **DINOv2-Small** (384-dim, fast)
- **MegaDescriptor-L-384** (1536-dim, trained on wildlife re-ID datasets)
- **MegaDescriptor-B-224** (768-dim, animal re-ID specialist)
- **ResNet50** (2048-dim, CNN baseline)
- **ConvNeXt Base** (1024-dim, modern CNN)
- **ConvNeXtV2 Base** (1024-dim, modern CNN v2 with MAE pretraining)
- **EfficientNet B3** (1536-dim, efficient CNN)
- **EfficientNetV2-RW-M** (hf-hub:timm/efficientnetv2_rw_m.agc_in1k, 2152-dim)

**Fixed Settings:**
- Loss: ArcFace (m=0.5, s=64)
- Dataset: JaguarCameraTrap/jaguars_camera_trap_0226-segmented_deduplicated (Hugging Face, FiftyOne export)
- Split field: closed_set_split
- Epochs: 50

Results logged to Wandb project: `camera-trap-reidentification`, group: `backbone_comparison`

## Setup

In [10]:
import sys
from pathlib import Path
import logging

# Add src to path
project_root = Path.cwd().parent / "camera-trap-footage"
sys.path.insert(0, str(project_root / "src"))

import fiftyone as fo

from jaguars.common.logging_utils import setup_logger
from jaguars.reidentification.config import get_default_config
from jaguars.reidentification.experiments import get_backbone_experiments
from jaguars.reidentification.training.train import run_processing as run_training
from jaguars.reidentification.wandb_results import fetch_latest_metrics_for_experiments

logger = setup_logger("backbone_experiments", level=logging.INFO)
print("✓ Imports successful")

✓ Imports successful


## Configuration

Configure base settings for all backbone experiments.

In [2]:
# Get default configuration
config = get_default_config()

# Run metadata for clear WandB separation
RUN_BATCH = "hf_0226_segmented_deduplicated_v2_cached"
DATASET_TAG = "dataset:hf_jaguars_camera_trap_0226_segmented_deduplicated"
SOURCE_TAG = "source:fiftyone_local_cache"

# Source dataset info
HF_REPO = "JaguarCameraTrap/jaguars_camera_trap_0226-segmented_deduplicated"
LOCAL_FO_DATASET_NAME = "JID_HF_0226_Segmented_Deduplicated_Cached"
LOCAL_HF_CACHE_DIR = project_root / "data" / "intermediate" / "hf_cache" / "jaguars_camera_trap_0226_segmented_deduplicated"

# Wandb settings
config.wandb.enabled = True
config.wandb.entity = "jaguars"
config.wandb.project = "camera-trap-reidentification"
config.wandb.tags = ["backbone_experiment", "subcenter_arcface_loss", DATASET_TAG, SOURCE_TAG, f"run_batch:{RUN_BATCH}"]

# Dataset settings (local FiftyOne cache)
config.dataset.source = "fiftyone"
config.dataset.fo_dataset_name = LOCAL_FO_DATASET_NAME
config.dataset.fo_split_field = "closed_set_split"
config.dataset.fo_patches_field = "sam3_segmentations"
config.dataset.fo_label_field = "ground_truth"
config.dataset.fo_embeddings_field = None

# Keep standard split names for downstream code
config.dataset.train_split = "train"
config.dataset.val_split = "val"
config.dataset.test_split = "test"

# Training settings
config.training.num_epochs = 50
config.training.loss_name = "subcenter_arcface"
config.model.arcface_margin = 0.5
config.model.arcface_scale = 64.0

print(f"✓ Base config loaded")
print(f"  Wandb project: {config.wandb.project}")
print(f"  Wandb tags: {config.wandb.tags}")
print(f"  HF repo (source): {HF_REPO}")
print(f"  Local HF cache dir: {LOCAL_HF_CACHE_DIR}")
print(f"  FiftyOne dataset: {config.dataset.fo_dataset_name}")
print(f"  Split field: {config.dataset.fo_split_field}")
print(f"  Label field: {config.dataset.fo_label_field}")
print(f"  Patches field: {config.dataset.fo_patches_field}")
print(f"  Loss: {config.training.loss_name}")
print(f"  ArcFace: m={config.model.arcface_margin}, s={config.model.arcface_scale}")
print(f"  Epochs: {config.training.num_epochs}")

✓ Base config loaded
  Wandb project: camera-trap-reidentification
  Wandb tags: ['backbone_experiment', 'subcenter_arcface_loss', 'dataset:hf_jaguars_camera_trap_0226_segmented_deduplicated', 'source:fiftyone_local_cache', 'run_batch:hf_0226_segmented_deduplicated_v2_cached']
  HF repo (source): JaguarCameraTrap/jaguars_camera_trap_0226-segmented_deduplicated
  Local HF cache dir: /sc/home/philipp.kolbe/JID/camera-trap-footage/camera-trap-footage/data/intermediate/hf_cache/jaguars_camera_trap_0226_segmented_deduplicated
  FiftyOne dataset: JID_HF_0226_Segmented_Deduplicated_Cached
  Split field: closed_set_split
  Label field: ground_truth
  Patches field: sam3_segmentations
  Loss: subcenter_arcface
  ArcFace: m=0.5, s=64.0
  Epochs: 50


In [3]:
# This notebook expects cache to be prepared by notebooks/prepare_fiftyone_cache.ipynb
if not fo.dataset_exists(LOCAL_FO_DATASET_NAME):
    raise ValueError(
        f"Local FiftyOne dataset '{LOCAL_FO_DATASET_NAME}' not found. "
        "Run notebooks/prepare_fiftyone_cache.ipynb first."
    )

dataset = fo.load_dataset(LOCAL_FO_DATASET_NAME)
images_view = dataset.select_group_slices("image") if dataset.group_field else dataset

print(f"✓ Using prepared local FiftyOne dataset: {LOCAL_FO_DATASET_NAME}")
print(f"  Total samples: {len(dataset)}")
print(f"  Image samples: {len(images_view)}")
print(f"  Group field: {dataset.group_field}")
print(f"  Splits: {images_view.count_values(config.dataset.fo_split_field)}")

✓ Using prepared local FiftyOne dataset: JID_HF_0226_Segmented_Deduplicated_Cached
  Total samples: 1998
  Image samples: 1998
  Group field: group
  Splits: {'train': 1632, 'val': 167, 'test': 199}


## Stage HF Export as Local FiftyOne Dataset

Download the Hugging Face FiftyOne export once, import it as a persistent local FiftyOne dataset, and reuse it across all runs.

## Load Experiments

In [4]:
# Get backbone experiments with our custom config
backbone_experiments = get_backbone_experiments(base_config=config)

# Ensure all runs carry the same dataset/run-batch tags and a unique prefix
for exp in backbone_experiments:
    exp.base_config.wandb.tags = list(dict.fromkeys(exp.base_config.wandb.tags + [DATASET_TAG, SOURCE_TAG, f"run_batch:{RUN_BATCH}"]))
    run_base = exp.base_config.wandb.run_name or exp.name
    exp.base_config.wandb.run_name = f"{RUN_BATCH}__{run_base}"

print(f"✓ {len(backbone_experiments)} backbone experiments configured:")
for exp in backbone_experiments:
    print(f"  - {exp.name}: {exp.description}")

✓ 14 backbone experiments configured:
  - backbone_vit_large_patch16_dinov3.lvd1689m: DINOv3 Large (1024-dim, latest self-supervised)
  - backbone_vit_base_patch16_dinov3.lvd1689m: DINOv3 Base (768-dim, efficient high-quality)
  - backbone_conservationxlabs_miewid-msv2: MiewID-MSv2 (wildlife re-ID specialist)
  - backbone_conservationxlabs_miewid-msv3: MiewID-MSv3 (wildlife re-ID specialist)
  - backbone_vit_large_patch14_dinov2.lvd142m: DINOv2 Large (1024-dim, best quality)
  - backbone_vit_base_patch14_dinov2.lvd142m: DINOv2 Base (768-dim, balanced)
  - backbone_vit_small_patch14_dinov2.lvd142m: DINOv2 Small (384-dim, fast)
  - backbone_hf-hub:BVRA_MegaDescriptor-L-384: MegaDescriptor Large 384 (animal re-ID specialist)
  - backbone_hf-hub:BVRA_MegaDescriptor-B-224: MegaDescriptor Base 224 (animal re-ID specialist)
  - backbone_resnet50: ResNet50 (25M params, CNN baseline)
  - backbone_convnext_base: ConvNeXt Base (modern CNN)
  - backbone_convnextv2_base.fcmae_ft_in22k_in1k: ConvNeX

## Run Experiments

Train each backbone with fixed ArcFace loss and compare results.

In [8]:
import gc
import torch

# Run all backbone experiments
results = {}

for experiment in backbone_experiments:
    logger.info(f"Running experiment: {experiment.name}")
    logger.info(f"  Description: {experiment.description}")
    logger.info(f"  Backbone: {experiment.base_config.backbone.name}")
    logger.info(f"  Embedding dim: {experiment.base_config.backbone.embedding_dim}")
    logger.info(f"  Tags: {experiment.base_config.wandb.tags}")
    logger.info(f"  Group: {experiment.group}")
    
    try:
        # Run training
        result = run_training(experiment.base_config)
        results[experiment.name] = result
        logger.info(f"✓ {experiment.name} completed")
    except Exception as e:
        logger.error(f"✗ {experiment.name} failed: {e}")
        results[experiment.name] = {"error": str(e)}
    finally:
        # Force memory cleanup between experiments to prevent OOM
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print(f"\n✓ All {len(backbone_experiments)} backbone experiments completed")

17:30:35 - jid_logger.backbone_experiments - INFO - Running experiment: backbone_hf-hub:BVRA_MegaDescriptor-L-384
17:30:35 - jid_logger.backbone_experiments - INFO -   Description: MegaDescriptor Large 384 (animal re-ID specialist)
17:30:35 - jid_logger.backbone_experiments - INFO -   Backbone: hf-hub:BVRA/MegaDescriptor-L-384
17:30:35 - jid_logger.backbone_experiments - INFO -   Embedding dim: 1536
17:30:35 - jid_logger.backbone_experiments - INFO -   Tags: ['backbone_experiment', 'subcenter_arcface_loss', 'dataset:hf_jaguars_camera_trap_0226_segmented_deduplicated', 'source:fiftyone_local_cache', 'run_batch:hf_0226_segmented_deduplicated_v2_cached']
17:30:35 - jid_logger.backbone_experiments - INFO -   Group: backbone_comparison
17:30:35 - jid_logger.reidentification.training - INFO - Starting re-identification training...
17:30:35 - jid_logger.reidentification.training - INFO - Dataset source: fiftyone
17:30:35 - jid_logger.reidentification.training - INFO - Backbone: hf-hub:BVRA/Me

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /sc/home/philipp.kolbe/.netrc.
wandb: Currently logged in as: hpi-philipp-kolbe to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


17:30:38 - jid_logger.reidentification.training - INFO - Loading dataset...
17:31:09 - jid_logger.reidentification.training - INFO - Dataset loaded:
17:31:09 - jid_logger.reidentification.training - INFO -   Train: 1632 samples
17:31:09 - jid_logger.reidentification.training - INFO -   Val: 167 samples
17:31:09 - jid_logger.reidentification.training - INFO -   Num classes: 175
17:31:09 - jid_logger.reidentification.training - INFO - Extracting embeddings with backbone...
Loading hf-hub:BVRA/MegaDescriptor-L-384 model...
Model loaded successfully
  Parameters: 195,198,516
  Embedding dimension: 1536


Val embeddings: 100%|██████████| 6/6 [00:10<00:00,  1.75s/it]

17:33:10 - jid_logger.reidentification.training - INFO - Embeddings extracted: (1632, 1536)
17:33:10 - jid_logger.reidentification.training - INFO - DataLoaders created:
17:33:10 - jid_logger.reidentification.training - INFO -   Train batches: 51
17:33:10 - jid_logger.reidentification.training - INFO -   Val batches: 6


...

17:45:24 - jid_logger.reidentification.training - INFO - Train Loss: 7.3329, Train Acc: 43.50%


17:45:24 - jid_logger.reidentification.training - INFO - Val Loss: 12.7788, Val Acc: 44.31%
17:45:24 - jid_logger.reidentification.training - INFO - Val mAP (identity-balanced): 0.5472
17:45:24 - jid_logger.reidentification.training - INFO - Val CMC@1: 0.6992, CMC@5: 0.8045
17:45:24 - jid_logger.reidentification.training - INFO - Learning Rate: 0.000100
17:45:25 - jid_logger.reidentification.training - INFO - Saved best model to data/models/reidentification/best_model.pt
17:45:25 - jid_logger.reidentification.training - INFO - 
Epoch 50/50


17:45:25 - jid_logger.reidentification.training - INFO - Train Loss: 7.0538, Train Acc: 45.53%
17:45:25 - jid_logger.reidentification.training - INFO - Val Loss: 12.5244, Val Acc: 45.51%
17:45:25 - jid_logger.reidentification.training - INFO - Val mAP (identity-balanced): 0.5443
17:45:25 - jid_logger.reidentification.training - INFO - Val CMC@1: 0.7068, CMC@5: 0.8045
17:45:25 - jid_logger.reidentification.training - INFO - Learning Rate: 0.000100
17:45:25 - jid_logger.reidentification.training - INFO - Saved checkpoint to data/models/reidentification/checkpoint_epoch_50.pt
17:45:25 - jid_logger.reidentification.training - INFO - ======================================================================
17:45:25 - jid_logger.reidentification.training - INFO - Training completed!
17:45:25 - jid_logger.reidentification.training - INFO - Best epoch: 49
17:45:25 - jid_logger.reidentification.training - INFO - Best val_map: 0.5472


epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/acc,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇██
train/batch_acc,▁▁▁▁▁▁▁▂▂▂▃▂▃▃▃▃▃▄▄▄▄▄▅▄▅▅▅▆▅▅▆▆▇▆▇▇▇▇█▇
train/batch_cls_loss,███▇▇▆▆▅▅▅▄▄▄▄▃▄▃▃▂▃▂▂▂▂▂▂▂▁▂▁▂▂▁▁▁▁▁▁▁▁
train/batch_loss,███▇▇▆▆▆▆▅▆▆▅▆▅▄▄▃▃▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▁▂▁▁▂▂
train/loss,█▇▇▇▆▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁
val/acc,▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇████
val/cmc@1,▁▁▁▁▁▂▁▂▂▂▃▃▃▃▄▄▄▅▅▅▅▅▆▆▇▆▆▆▇▇▇▇▇▇▇█████
val/cmc@10,▁▁▂▁▁▂▂▂▂▃▄▄▄▅▅▆▆▆▆▆▆▇▆▆▇▆▆▇▆▇▇▇▇▇▇█▇▇▇█
+10,...


17:45:26 - jid_logger.backbone_experiments - INFO - ✓ backbone_hf-hub:timm_efficientnetv2_rw_m.agc_in1k completed

✓ All 7 backbone experiments completed


## Results Summary

Compare backbone performance on validation set.

In [11]:
# Print summary of latest W&B results
import pandas as pd

MODEL_NAME_MAP = {
    "vit_large_patch16_dinov3.lvd1689m": "DINOv3-Large",
    "vit_base_patch16_dinov3.lvd1689m": "DINOv3-Base",
    "conservationxlabs/miewid-msv2": "MiewID-MSv2",
    "conservationxlabs/miewid-msv3": "MiewID-MSv3",
    "vit_large_patch14_dinov2.lvd142m": "DINOv2-Large",
    "vit_base_patch14_dinov2.lvd142m": "DINOv2-Base",
    "vit_small_patch14_dinov2.lvd142m": "DINOv2-Small",
    "hf-hub:BVRA/MegaDescriptor-L-384": "MegaDescriptor-L-384",
    "hf-hub:BVRA/MegaDescriptor-B-224": "MegaDescriptor-B-224",
    "resnet50": "ResNet50",
    "convnext_base": "ConvNeXt-Base",
    "convnextv2_base.fcmae_ft_in22k_in1k": "ConvNeXtV2-Base",
    "efficientnet_b3": "EfficientNet-B3",
    "hf-hub:timm/efficientnetv2_rw_m.agc_in1k": "EfficientNetV2-RW-M",
}

experiment_lookup = {exp.name: exp.base_config.backbone.name for exp in backbone_experiments}
wandb_results = fetch_latest_metrics_for_experiments(
    experiments=backbone_experiments,
    entity=config.wandb.entity,
    project=config.wandb.project,
    additional_tags=[DATASET_TAG, SOURCE_TAG, f"run_batch:{RUN_BATCH}"],
)

summary_data = []
for exp in backbone_experiments:
    exp_name = exp.name
    result = wandb_results.get(exp_name, {"error": "Missing W&B result"})
    backbone_name = experiment_lookup.get(exp_name, exp_name)
    display_name = MODEL_NAME_MAP.get(backbone_name, backbone_name)
    if "error" in result:
        summary_data.append({
            "Backbone": display_name,
            "mAP": "ERROR",
            "CMC@1": "ERROR",
            "mAP (>=9 total)": "ERROR",
            "Status": result["error"],
        })
    else:
        map_val = result.get("map", "N/A")
        cmc1 = result.get("cmc@1", "N/A")
        map_9 = result.get("map_min_total_9", "N/A")
        summary_data.append({
            "Backbone": display_name,
            "mAP": f"{map_val:.4f}" if isinstance(map_val, (int, float)) else str(map_val),
            "CMC@1": f"{cmc1:.4f}" if isinstance(cmc1, (int, float)) else str(cmc1),
            "mAP (>=9 total)": f"{map_9:.4f}" if isinstance(map_9, (int, float)) else str(map_9),
            "Status": "✓ Loaded from W&B",
        })

summary_df = pd.DataFrame(summary_data)
print("\n=== Backbone Comparison Results (Latest W&B Runs) ===\n")
print(summary_df.to_string(index=False))
print(f"\nView detailed results at: https://wandb.ai/{config.wandb.entity}/{config.wandb.project}")


=== Backbone Comparison Results (Latest W&B Runs) ===

            Backbone   mAP CMC@1 mAP (>=9 total)                                                 Status
MegaDescriptor-L-384 ERROR ERROR           ERROR No supported metrics found in selected W&B run summary
MegaDescriptor-B-224 ERROR ERROR           ERROR No supported metrics found in selected W&B run summary
            ResNet50 ERROR ERROR           ERROR No supported metrics found in selected W&B run summary
       ConvNeXt-Base ERROR ERROR           ERROR No supported metrics found in selected W&B run summary
     ConvNeXtV2-Base ERROR ERROR           ERROR No supported metrics found in selected W&B run summary
     EfficientNet-B3 ERROR ERROR           ERROR No supported metrics found in selected W&B run summary
 EfficientNetV2-RW-M ERROR ERROR           ERROR No supported metrics found in selected W&B run summary

View detailed results at: https://wandb.ai/jaguars/camera-trap-reidentification


## Export for Report

Generate report-ready artifacts (CSV, LaTeX table, PNG figure).

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

MODEL_NAME_MAP = {
    "vit_large_patch16_dinov3.lvd1689m": "DINOv3-Large",
    "vit_base_patch16_dinov3.lvd1689m": "DINOv3-Base",
    "conservationxlabs/miewid-msv2": "MiewID-MSv2",
    "conservationxlabs/miewid-msv3": "MiewID-MSv3",
    "vit_large_patch14_dinov2.lvd142m": "DINOv2-Large",
    "vit_base_patch14_dinov2.lvd142m": "DINOv2-Base",
    "vit_small_patch14_dinov2.lvd142m": "DINOv2-Small",
    "hf-hub:BVRA/MegaDescriptor-L-384": "MegaDescriptor-L-384",
    "hf-hub:BVRA/MegaDescriptor-B-224": "MegaDescriptor-B-224",
    "resnet50": "ResNet50",
    "convnext_base": "ConvNeXt-Base",
    "convnextv2_base.fcmae_ft_in22k_in1k": "ConvNeXtV2-Base",
    "efficientnet_b3": "EfficientNet-B3",
    "hf-hub:timm/efficientnetv2_rw_m.agc_in1k": "EfficientNetV2-RW-M",
}

experiment_lookup = {exp.name: exp.base_config.backbone.name for exp in backbone_experiments}

# Build summary_df from latest W&B runs if this cell is run standalone
if "summary_df" not in globals():
    wandb_results = fetch_latest_metrics_for_experiments(
        experiments=backbone_experiments,
        entity=config.wandb.entity,
        project=config.wandb.project,
        additional_tags=[DATASET_TAG, SOURCE_TAG, f"run_batch:{RUN_BATCH}"],
    )

    summary_data = []
    for exp in backbone_experiments:
        exp_name = exp.name
        result = wandb_results.get(exp_name, {"error": "Missing W&B result"})
        backbone_name = experiment_lookup.get(exp_name, exp_name)
        display_name = MODEL_NAME_MAP.get(backbone_name, backbone_name)
        if "error" in result:
            summary_data.append({
                "Backbone": display_name,
                "mAP": "ERROR",
                "CMC@1": "ERROR",
                "mAP (>=9 total)": "ERROR",
                "Status": result["error"],
            })
        else:
            map_val = result.get("map", "N/A")
            cmc1 = result.get("cmc@1", "N/A")
            map_9 = result.get("map_min_total_9", "N/A")
            summary_data.append({
                "Backbone": display_name,
                "mAP": f"{map_val:.4f}" if isinstance(map_val, (int, float)) else str(map_val),
                "CMC@1": f"{cmc1:.4f}" if isinstance(cmc1, (int, float)) else str(cmc1),
                "mAP (>=9 total)": f"{map_9:.4f}" if isinstance(map_9, (int, float)) else str(map_9),
                "Status": "✓ Loaded from W&B",
            })
    summary_df = pd.DataFrame(summary_data)

run_batch = globals().get("RUN_BATCH", "manual_run")
output_dir = Path("notebooks/data/results/figures/report") / run_batch / "backbone"
output_dir.mkdir(parents=True, exist_ok=True)

# Save tabular artifacts
csv_path = output_dir / "backbone_summary.csv"
tex_path = output_dir / "backbone_summary.tex"
summary_df.to_csv(csv_path, index=False)
summary_df.to_latex(tex_path, index=False)

# Save figure
plot_df = summary_df.copy()
plot_df["mAP_numeric"] = pd.to_numeric(plot_df["mAP"], errors="coerce")
plot_df = plot_df.dropna(subset=["mAP_numeric"]).sort_values("mAP_numeric", ascending=False)

fig_path = output_dir / "backbone_map_bar.png"
plt.figure(figsize=(12, max(4, 0.4 * len(plot_df))))
plt.barh(plot_df["Backbone"], plot_df["mAP_numeric"])
plt.gca().invert_yaxis()
plt.xlabel("mAP")
plt.title("Backbone Comparison")
plt.tight_layout()
plt.savefig(fig_path, dpi=300)
plt.close()

print(f"Saved CSV: {csv_path}")
print(f"Saved LaTeX table: {tex_path}")
print(f"Saved figure: {fig_path}")